# Robustly Optimized BERT Pretraining Approach

RoBERTa improves BERT with new pretraining objectives, demonstrating BERT was undertrained and training design is important. The pretraining objectives include dynamic masking, sentence packing, larger batches and a byte-level BPE tokenizer. (https://huggingface.co/docs/transformers/model_doc/roberta)

In [12]:
import torch
import pandas as pd
import numpy as np

# huggingface
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained("roberta-base")


Using device: cuda


In [13]:
# Load test set
# it only removed the escape characters 
train_df = pd.read_csv("trainset.csv")
val_df = pd.read_csv("valset.csv")
train_df['comment_text'] = train_df['comment_text'].astype(str)
val_df['comment_text'] = val_df['comment_text'].astype(str)

# reduce the size of the train due to memory issue
train_df = train_df.sample(frac=0.5, random_state=42)


In [14]:
# Configs
MODEL_NAME = "FacebookAI/roberta-base"
MAX_LEN = 150
BATCH_SIZE = 16 # for pytorch
LEARNING_RATE = 2e-5 # RoBERTa requires a much lower LR than CNN (which used 0.001)
EPOCHS = 3
LABEL_COLUMNS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
# sigmoid activation function is used for binary classification
# threshold is used to classify the output as 0 or 1
SIGMOID_THRESHOLD = 0.5

# Tokenization

In [15]:
# Convert to Huggingface Dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess_data(batch):
    # Tokenize the text
    encoding = tokenizer(
        batch["comment_text"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_LEN
    )
    
    # Create a list of float labels for each example in the batch
    # The model expects a FloatTensor of shape (batch_size, num_labels)
    labels_matrix = []
    for i in range(len(batch["comment_text"])):
        row_labels = [float(batch[col][i]) for col in LABEL_COLUMNS]
        labels_matrix.append(row_labels)
        
    encoding["labels"] = labels_matrix
    return encoding

# Apply preprocessing
encoded_train = train_ds.map(preprocess_data, batched=True, remove_columns=list(train_df.columns))
encoded_val = val_ds.map(preprocess_data, batched=True, remove_columns=list(val_df.columns))

# Set format for PyTorch
encoded_train.set_format("torch")
encoded_val.set_format("torch")

Map:   0%|          | 0/51062 [00:00<?, ? examples/s]

Map:   0%|          | 0/25532 [00:00<?, ? examples/s]

# RoBERTa Model

In [17]:
import numpy as np
import torch
import os
from sklearn.metrics import f1_score
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# Load Model
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification",
    device_map="auto" 
)

# Training Arguments
args = TrainingArguments(
    output_dir="./roberta_toxic_results",
    learning_rate=LEARNING_RATE,
    
    # Use 'eval_strategy' (newer versions) 
    eval_strategy="epoch",  
    save_strategy="epoch",
    
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    
    logging_steps=10, # Update progress bar every 10 steps
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # logits is a numpy array here
    probs = 1 / (1 + np.exp(-logits))
    
    preds = (probs >= SIGMOID_THRESHOLD).astype(int)
    
    # Metrics
    f1_macro = f1_score(labels, preds, average='macro')
    
    return {'f1_macro': f1_macro}

# Initialize Trainer
trainer = Trainer(
    model=roberta_model,    
    args=args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train and Save
save_path = "roberta_toxic_classifier.pth"

if os.path.exists(save_path):
    print(f"Loading pre-trained model from {save_path}")
    state_dict = torch.load(save_path, map_location=device)
    roberta_model.load_state_dict(state_dict)
else:
    print("Starting training...")
    trainer.train()
    torch.save(roberta_model.state_dict(), save_path)
    print(f"Model state dictionary saved to {save_path}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\cskok\AppData\Local\Temp\ipykernel_68668\3479083919.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training...


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.093200,0.045150,0.395172
2,0.050400,0.043284,0.601576
3,0.011900,0.047804,0.665001


Model state dictionary saved to roberta_toxic_classifier.pth


# Test the Model with Test Set

In [19]:
# Load Data
test_df = pd.read_csv("testset_transformer.csv")
test_df['comment_text'] = test_df['comment_text'].astype(str)

# Convert to Hugging Face Dataset
test_ds = Dataset.from_pandas(test_df)

# Apply Tokenization 
encoded_test = test_ds.map(preprocess_data, batched=True, remove_columns=list(test_df.columns))
encoded_test.set_format("torch")

# predict
output = trainer.predict(encoded_test)
logits = output.predictions

# sigmoid
probs = 1 / (1 + np.exp(-logits))
pred_labels = (probs >= 0.5).astype(int)

# Save to DataFrame 
test_pred_df = test_df.copy()
test_pred_df[LABEL_COLUMNS] = pred_labels

test_pred_df.head()

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,"Geez, are you forgetful! We've already discuss...",0,0,0,0,0,0
1,Carioca RFA Thanks for your support on my requ...,0,0,0,0,0,0
2,""" Birthday No worries, It's what I do ;)Enjoy ...",0,0,0,0,0,0
3,Pseudoscience category? I'm assuming that this...,0,0,0,0,0,0
4,"(and if such phrase exists, it would be provid...",0,0,0,0,0,0


In [22]:
from sklearn.metrics import multilabel_confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

def create_confusion_matrix(true_df, pred_df):
    """ 
    true_df: Dataframe that has true labels 
    pred_df: Dataframe that has predicted labels
    """
    y_true = true_df[LABEL_COLUMNS].values
    y_pred = pred_df[LABEL_COLUMNS].values

    # The output is a 3D NumPy array: (labels, 2, 2)
    mcm = multilabel_confusion_matrix(y_true, y_pred) 

    print("--- RoBERTa Multilabel Confusion Matrices ---")
    for i, col in enumerate(LABEL_COLUMNS):
        matrix = mcm[i]
        TN = matrix[0, 0]
        FP = matrix[0, 1]
        FN = matrix[1, 0]
        TP = matrix[1, 1]
        
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        precision = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
        recall = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
        
        print(f"{col:15} → F1: {f1:.4f}, Acc: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f} | TP:{TP} TN:{TN} FP:{FP} FN:{FN}")

# Run the evaluation
create_confusion_matrix(test_df, test_pred_df)

--- RoBERTa Multilabel Confusion Matrices ---
toxic           → F1: 0.8227, Acc: 0.9672, Precision: 0.8518, Recall: 0.7955 | TP:2431 TN:28436 FP:423 FN:625
severe_toxic    → F1: 0.4055, Acc: 0.9904, Precision: 0.5417, Recall: 0.3240 | TP:104 TN:31506 FP:88 FN:217
obscene         → F1: 0.8285, Acc: 0.9821, Precision: 0.8558, Recall: 0.8029 | TP:1377 TN:29968 FP:232 FN:338
threat          → F1: 0.4853, Acc: 0.9978, Precision: 0.5323, Recall: 0.4459 | TP:33 TN:31812 FP:29 FN:41
insult          → F1: 0.7449, Acc: 0.9758, Precision: 0.7982, Recall: 0.6983 | TP:1127 TN:30016 FP:285 FN:487
identity_hate   → F1: 0.5040, Acc: 0.9923, Precision: 0.6188, Recall: 0.4252 | TP:125 TN:31544 FP:77 FN:169
